# CRISP-DM Phase 3: Data Preparation
## Data Cleaning & Export — Crédit Nationale Azur
This notebook addresses all quality issues identified during EDA:
duplicates, missing values, and outliers. The cleaned dataset is exported
as a pickle file using joblib to guarantee a static, reproducible input
for all modelling notebooks.

CRITICAL: Categorical encoding, standardisation, and feature selection are
deliberately deferred to the modelling phase to prevent data leakage.

In [6]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv('../data/personal-loan.csv')
print(f"Original shape: {df.shape}")
print(f"Duplicate rows found: {df.duplicated().sum()}")

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# The 'customer_id' column is a unique identifier and does not provide predictive value, so we drop it to prevent any potential data leakage or overfitting.
df = df.drop('customer_id', axis=1)

print(f"Shape after duplicate removal and customer_id drop: {df.shape}")

Original shape: (6000, 13)
Duplicate rows found: 0
Shape after duplicate removal and customer_id drop: (6000, 12)


## 4.1 Missing Value Imputation
Missing values exist in: Age, Years of Experience, Family Size, Income.
Median imputation is chosen over mean because income and related variables
are right-skewed (confirmed in EDA). Mean imputation would artificially
inflate imputed values, biasing feature distributions used in model training.
Record deletion is rejected — it would discard valid data in other columns
and unnecessarily reduce dataset size.

In [2]:
print("Missing values before imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

for col in df.select_dtypes(include='number').columns:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)  # reassign instead of inplace=True
        print(f"  Imputed '{col}' with median: {median_val:.2f}")

print(f"\nMissing values after imputation: {df.isnull().sum().sum()}")

Missing values before imputation:
age               380
yrs_experience    364
family_size       385
income            377
dtype: int64
  Imputed 'age' with median: 45.00
  Imputed 'yrs_experience' with median: 21.00
  Imputed 'family_size' with median: 2.00
  Imputed 'income' with median: 82.00

Missing values after imputation: 0


## 4.2 Outlier Treatment — IQR Capping
Continuous variables are treated using the IQR method.
Values outside 1.5 × IQR bounds are replaced with the column median.

Rationale for capping over deletion: preserves dataset size.
Rationale for median replacement over boundary-value replacement:
avoids introducing new extreme values at the IQR fence.

This step is critical for SVM, which computes spatial decision boundaries
and is disproportionately distorted by extreme values in high-magnitude
features like Income and Mortgage Amount.

In [3]:
continuous_cols = ['age', 'yrs_experience', 'family_size',
                   'income', 'mortgage_amt', 'credit_card_spend']

for col in continuous_cols:
    median_val     = df[col].quantile(0.50)
    percentile_25  = df[col].quantile(0.25)
    percentile_75  = df[col].quantile(0.75)
    iqr            = percentile_75 - percentile_25

    lower = percentile_25 - (1.5 * iqr)
    upper = percentile_75 + (1.5 * iqr)

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.where((df[col] < lower) | (df[col] > upper), median_val, df[col])
    print(f"{col}: {outliers} outliers replaced with median ({median_val:.2f})")

print(f"\nFinal shape: {df.shape}")
print(f"Final missing values: {df.isnull().sum().sum()}")

age: 66 outliers replaced with median (45.00)
yrs_experience: 76 outliers replaced with median (21.00)
family_size: 34 outliers replaced with median (2.00)
income: 99 outliers replaced with median (82.00)
mortgage_amt: 79 outliers replaced with median (0.00)
credit_card_spend: 1089 outliers replaced with median (0.00)

Final shape: (6000, 12)
Final missing values: 0


## 4.3 Final Verification
Confirm dtypes and zero missing values before pickling.

In [4]:
print("Final column dtypes:")
print(df.dtypes)
print(f"\nFinal missing values per column:\n{df.isnull().sum()}")
print(f"\nFinal dataset shape: {df.shape}")

Final column dtypes:
age                   float64
yrs_experience        float64
family_size           float64
education_level           str
income                float64
mortgage_amt          float64
credit_card_acct          str
credit_card_spend     float64
share_trading_acct      int64
fixed_deposit_acct      int64
online_acct               str
personal_loan             str
dtype: object

Final missing values per column:
age                   0
yrs_experience        0
family_size           0
education_level       0
income                0
mortgage_amt          0
credit_card_acct      0
credit_card_spend     0
share_trading_acct    0
fixed_deposit_acct    0
online_acct           0
personal_loan         0
dtype: int64

Final dataset shape: (6000, 12)


## 4.4 Export Cleaned Dataset via joblib
The cleaned dataset is pickled using joblib to guarantee a static,
unaltered input baseline for all subsequent modelling notebooks.
This prevents accidental drift between notebooks and ensures full
reproducibility across all model configurations.

In [5]:
joblib.dump(df, '../data/cleaned_data.pkl')
print("cleaned_data.pkl saved to ../data/")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

cleaned_data.pkl saved to ../data/
Rows: 6000 | Columns: 12
